## Data Exploration / Understanding

In [3]:
# view the three datasets
import pandas as pd
import sys
from pathlib import Path

sys.path.append(str(Path("..").resolve()))

TRANSCRIPTS_PATH = Path("..") / "data" / "transcripts_cleaned.csv" # not using this in this notebook

In [6]:
transcript_df = pd.read_csv(TRANSCRIPTS_PATH)
print(transcript_df.shape)
transcript_df.head()

(157, 6)


,Record-ID,Class,Transcript_PFT,Transcript_CTD,Transcript_SFT,Label
0,Process-rec-001,MCI,"people, partner, plate, platter, pants, porter...",NaN,"<pause_medium> giraffe, kangaroo, lion, tiger,...",1
1,Process-rec-002,MCI,"<pause_short> pipe, plane, people <pause_mediu...",<pause_medium> there’s a lad stood on the stoo...,"<pause_short> dogs, cats, birds <pause_short> ...",1
2,Process-rec-003,MCI,"um <pause_short> purple, pale, placid <pause_s...","<pause_medium> um, the picture is of a kitchen...","cow, bull, ewe, ram, chicken, goose, um <sigh>...",1
3,Process-rec-004,MCI,plank <pause_short> pool <pause_short> swimmin...,"a mother presumably, or a fe, an adult female ...",um <pause_short> impala <pause_short> er cheet...,1
4,Process-rec-005,MCI,"it’s er pillock, er post box, er pyracanthas, ...","‘50s style er scene of domestic um confusion, ...","dog, cat, giraffe, wallaby, kangaroo, tortoise...",1


In [9]:
DATASET_PATH = Path("..") / "data" / "dementia_data.csv"
dataset_df = pd.read_csv(DATASET_PATH)
print(dataset_df.shape)
dataset_df.head()

(157, 25)


,Record-ID,TrainOrDev,Class,Gender,Age,Converted-MMSE,Transcript_PFT,Transcript_CTD,Transcript_SFT,Class_label,...,filler_count,token_count,type_count,type_token_ratio,ma_ttr,brunets_index,content_density,repetitions,sentence_count,average_words_per_sentence
0,Process-rec-001,train,MCI,1,62,25.0,"Pat: People, partner, plate, platter, pants, p...",NaN,"Pat: (3 seconds) Giraffe, kangaroo, lion, tige...",1,...,1,0,0,0.000000,0.000000,0.000000,0.000000,{},0,0.000000
1,Process-rec-002,dev,MCI,1,61,25.0,"Pat: (1 second) Pipe, plane, people (5 seconds...",Pat: (4 seconds) There’s a lad stood on the st...,"Pat: (1 second) Dogs, cats, birds (1 second) m...",1,...,2,76,46,0.605263,1.000000,1.487174,0.421053,"{'there': 2, 's': 7, 'a': 5, 'stood': 2, 'on':...",5,15.200000
2,Process-rec-003,train,MCI,0,62,29.0,"Pat: Um (1 second) purple, pale, placid (1 sec...","Pat: (3 seconds) Um, the picture is of a kitch...","Pat: Cow, bull, ewe, ram, chicken, goose, um (...",1,...,6,150,84,0.560000,0.995495,1.620714,0.433333,"{'the': 10, 'is': 5, 'of': 2, 'a': 10, 'there'...",7,21.428571
3,Process-rec-004,dev,MCI,0,67,29.0,Pat: Plank (1 second) pool (1 second) swimming...,"Pat: A mother presumably, or a fe, an adult fe...",Pat: Um (1 second) impala (1 second) er cheeta...,1,...,4,162,88,0.543210,1.000000,1.675909,0.425926,"{'a': 5, 'or': 2, 'an': 4, 'adult': 3, 'female...",4,40.500000
4,Process-rec-005,train,MCI,1,65,27.0,"Pat: It’s er pillock, er post box, er Pyracant...",Pat: ‘50s style er scene of domestic um confus...,"Pat: Dog, cat, giraffe, wallaby, kangaroo, tor...",1,...,4,44,36,0.818182,1.000000,1.057222,0.500000,"{'s': 3, 'of': 4, 'the': 3, 'and': 2}",1,44.000000


In [10]:
# all the 25 columns
print(dataset_df.columns)

Index(['Record-ID', 'TrainOrDev', 'Class', 'Gender', 'Age', 'Converted-MMSE',
       'Transcript_PFT', 'Transcript_CTD', 'Transcript_SFT', 'Class_label',
       'total_seconds', 'Parentheses_Content', 'CTD_Cleaned', 'found_fillers',
       'filler_list', 'filler_count', 'token_count', 'type_count',
       'type_token_ratio', 'ma_ttr', 'brunets_index', 'content_density',
       'repetitions', 'sentence_count', 'average_words_per_sentence'],
      dtype='object')


In [13]:
# first row of dementia_data
print(dataset_df.head(1))

         Record-ID TrainOrDev Class  Gender Age  Converted-MMSE  \
0  Process-rec-001      train   MCI       1  62            25.0   

                                      Transcript_PFT Transcript_CTD  \
0  Pat: People, partner, plate, platter, pants, p...            NaN   

                                      Transcript_SFT  Class_label  ...  \
0  Pat: (3 seconds) Giraffe, kangaroo, lion, tige...            1  ...   

   filler_count token_count type_count type_token_ratio ma_ttr  brunets_index  \
0             1           0          0              0.0    0.0            0.0   

   content_density  repetitions  sentence_count  average_words_per_sentence  
0              0.0           {}               0                         0.0  

[1 rows x 25 columns]


In [30]:
# imports
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

In [24]:
# let's make a copy of the dataset_df and split into train_X, test_X, train_Y, text_Y
# let's also decide which features to include and which features to change

df = dataset_df.copy()
# y = numeric class label (0 = HC, 1 = MCI, 2 = Dementia)
y = df["Class_label"]

# fix error in Age and convert from Sting to Numeric
df["Age"] = df["Age"].replace("66*", "66")
df["Age"] = pd.to_numeric(df["Age"], errors="coerce")

# combine transcripts into one text field
df["all_text"] = (
    df["Transcript_PFT"].fillna("") + " " +
    df["Transcript_SFT"].fillna("") + " " +
    df["Transcript_CTD"].fillna("")
)

# choose numeric features you want to include
numeric_features = [
    "Converted-MMSE",
    "Age",
    "filler_count",
    "token_count",
    "type_count",
    "type_token_ratio",
    "ma_ttr",
    "brunets_index",
    "content_density",
    "sentence_count",
    "average_words_per_sentence",
]

categorical_features = ["Gender"]

# dropped: Record-ID, TrainOrDev, Class, repetitions (maybe engineer this later)

In [23]:
# See if any NaN or incorrect numeric values

for col in numeric_features:
    print(f"\nColumn: {col}")
    print(df[col].unique())


Column: Converted-MMSE
[25.         29.         27.         26.         27.36231884 28.
 19.         22.         23.         30.         20.         24.        ]

Column: Age
['62' '61' '67' '65' '83' '68' '77' '56' '60' '72' '74' '66' '45' '75'
 '70' '80' '71' '69' '73' '86' '51' '79' '55' '52' '57' '82' '76' '78'
 '37' '23' '49' '64' '24' '36' '30' '59' '63' '94' '84' '50' '87' '66*'
 '58' '27']

Column: Gender
[1 0 2]

Column: filler_count
[ 1  2  6  4  3  7  5  9  8 40 10 14 26 13 12 21 18 11 16 39 22 17 15 29
 28 38]

Column: token_count
[  0  76 150 162  44 122 219 100 184 307 109 161 351 224 188 143  66 134
 306  52  69 222 135  80 334 130  83 212  92  98 115 152 283  81 243 239
 180 247  77 138 406 268  49 118  71 145 119 482 198 156 221 269 160 149
 167 140 129  90 127 237 117 120 128 309 213 133 267  48  47 197  65 216
 113 257 200 114  87 193 154 111 340 187 123 179 367  63 225  78  26  82
 182 254  86 391 195 311 166 317 168  88  22  25  61 206 176  95  67  39
 153  74 233

In [34]:
# X = text + numeric columns + cat (gender)
X = df[[text_feature] + numeric_features + categorical_features]

In [35]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

## Preprocessing: TF-IDF for text, scaling for numeric

In [36]:
text_feature = "all_text"

text_transformer = TfidfVectorizer(
    ngram_range=(1, 2),   # unigrams + bigrams
    min_df=2,             # ignore super-rare words
    max_features=10000,   # cap vocab size (tweak if you want)
)

numeric_transformer = StandardScaler()
categorical_transformer = OneHotEncoder(handle_unknown="ignore")

preprocess = ColumnTransformer(
    transformers=[
        ("text", text_transformer, text_feature),
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ],
    remainder="drop",
)

## K-Fold

In [49]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_scores = cross_val_score(
    clf,
    X_train,          # IMPORTANT: only training data
    y_train,
    cv=cv,
    scoring="f1_macro"   # or "accuracy", "f1_weighted", etc.
)

print("CV scores:", cv_scores)
print("Mean CV:", cv_scores.mean(), "±", cv_scores.std())

CV scores: [0.63729246 0.31800766 0.5        0.34188034 0.4040404 ]
Mean CV: 0.4402441747269334 ± 0.11689885266282973


In [43]:
# Standard deviation = 0.1169 which means that my model performance swings a LOT depending on the split

## Creating the Logistic Regression Model + Training It

In [50]:
log_reg = LogisticRegression(
    solver="lbfgs",           # liblinear or lbfgs # good for multinomial
    max_iter=1000,
    class_weight="balanced",  # handle HC/MCI/Dementia imbalance
    n_jobs=-1,                # use all cores
)
clf = Pipeline(steps=[
    ("preprocess", preprocess),
    ("logreg", log_reg),
])

In [51]:
clf.fit(X_train, y_train)

,steps,"[('preprocess', ...), ('logreg', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('text', ...), ('num', ...), ...]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


## Evaluating the Model

In [52]:
y_pred = clf.predict(X_test)

print("Classification report:")
print(classification_report(y_test, y_pred))

print("Confusion matrix:")
print(confusion_matrix(y_test, y_pred))

Classification report:
              precision    recall  f1-score   support

           0       0.72      0.76      0.74        17
           1       0.56      0.42      0.48        12
           2       0.20      0.33      0.25         3

    accuracy                           0.59        32
   macro avg       0.49      0.50      0.49        32
weighted avg       0.61      0.59      0.60        32

Confusion matrix:
[[13  3  1]
 [ 4  5  3]
 [ 1  1  1]]


## Optional: K Fold training and Evaluating on best Fold

In [53]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

param_grid = {
    "logreg__C": [0.01, 0.1, 1, 10],
    "logreg__solver": ["lbfgs"],
}

grid = GridSearchCV(
    clf,
    param_grid,
    cv=cv,
    scoring="f1_macro",   # or "accuracy"
    n_jobs=-1,
)

grid.fit(X_train, y_train)

print("Best params:", grid.best_params_)
print("Best CV score:", grid.best_score_)
best_clf = grid.best_estimator_

Best params: {'logreg__C': 10, 'logreg__solver': 'lbfgs'}
Best CV score: 0.46359172581816477


In [54]:
y_pred = best_clf.predict(X_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.79      0.88      0.83        17
           1       0.64      0.58      0.61        12
           2       0.00      0.00      0.00         3

    accuracy                           0.69        32
   macro avg       0.48      0.49      0.48        32
weighted avg       0.66      0.69      0.67        32

